# Medical Disease Diagnosis from Image of Patients Symptoms Report (Federated Learning using NLP and BERT Transformer with Explainability)

## 1. Importing Required Libraries

In [ ]:
import pandas as pd
import torch
import numpy as np
import joblib
import shap

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from datasets import Dataset

In [2]:
import os
os.environ["WANDB_DISABLED"] = "true"

## 2. Loading Datasets


In [3]:
train_dfs = [
    pd.read_csv('/kaggle/input/medical-federated1/gretalai_train.csv'),
    pd.read_csv('/kaggle/input/medical-federated1/venetis_train.csv'),
    pd.read_csv('/kaggle/input/medical-federated1/Symptom2Disease_train.csv')
]

test_dfs = [
    pd.read_csv('/kaggle/input/medical-federated1/gretalai_test.csv'),
    pd.read_csv('/kaggle/input/medical-federated1/venetis_test.csv'),
    pd.read_csv('/kaggle/input/medical-federated1/Symptom2Disease_test.csv')
]

## 3. Display Sample Data

In [6]:
train_dfs[0]

,description,disease
0,I've been having a lot of pain in my neck and ...,Cervical Spondylosis
1,I have a rash on my face that is getting worse...,Impetigo
2,I have been urinating blood. I sometimes feel ...,Urinary Tract Infection
3,I have been having trouble with my muscles and...,Arthritis
4,I have been feeling really sick. My body hurts...,Dengue
...,...,...
848,My veins are bulging and painful. I can't stan...,Varicose Veins
849,I have been having headaches for a while now. ...,Migraine
850,I have a rash on my face that is very painful ...,Impetigo
851,I have a stuffy nose and nasal congestion. I s...,Allergy


In [9]:
test_dfs[0]

,description,disease
0,I have a burning sensation in my stomach that ...,Peptic Ulcer Disease
1,I have a hard time swallowing and I feel like ...,Peptic Ulcer Disease
2,"I've been having headaches and migraines, and ...",Drug Reaction
3,I'm sweating a lot and can't catch my breath. ...,Pneumonia
4,"I've been scratching myself a lot lately, and ...",Fungal Infection
...,...,...
207,I have been experiencing muscle pain that make...,Dengue
208,"I have red, irritated skin on my arms, face, a...",Psoriasis
209,"I've been having a hard time breathing, and I'...",Bronchial Asthma
210,I've been coughing a lot for a few days now. I...,Bronchial Asthma


In [7]:
train_dfs[1]

,description,disease
0,dr. i feel a strange and powerful pain inside ...,heart hurts
1,my head ache since i woke up this morning.,head ache
2,the pain feels like it's right below the skin,muscle pain
3,i have a blurry vision after my head was hit y...,blurry vision
4,there is a tingling sensation in my neck.,neck pain
...,...,...
5323,pain in the large neck,neck pain
5324,my son nicked his neck with an old razor and t...,infected wound
5325,my ear hurts and it's worse when i swallow. m...,ear ache
5326,i got acne when i ate chili,acne


In [10]:
test_dfs[1]

,description,disease
0,i have eruptions on my face that come and go .,acne
1,i feel increased heart rate with prick,heart hurts
2,i have a hair shortage,hair falling out
3,i have a sharp pain in my lower stomach.,stomach ache
4,my calves feel like they are tight as knots an...,muscle pain
...,...,...
662,i must see a doctor i have an open wound,open wound
663,i have a wound between my toes that gets bette...,open wound
664,i've had this cough for two weeks.,cough
665,my knee feels weak and it gave way the other d...,knee pain


In [8]:
train_dfs[2]

,disease,description
0,Chicken pox,I'm feeling really nauseous and uneasy. I'm no...
1,Common Cold,"I've been feeling awful, with a lot of congest..."
2,Common Cold,I've been coughing a lot and finding it diffic...
3,Dimorphic Hemorrhoids,I've been having a lot of problems using the r...
4,drug reaction,In addition to experiencing a change in taste ...
...,...,...
955,urinary tract infection,"My spirits have been incredibly low, and my pe..."
956,Psoriasis,"There is a silver like dusting on my skin, esp..."
957,Migraine,I have been feeling hungry all the time and ha...
958,Fungal infection,A rash that appears to be developing throughou...


In [11]:
test_dfs[2]

,disease,description
0,Pneumonia,"I've been experiencing chills, feel really exh..."
1,Malaria,"I've had intense itching all over my body, acc..."
2,Dengue,I have been feeling extremely tired and fatigu...
3,Dimorphic Hemorrhoids,I'm having a lot of trouble with my bowel move...
4,Arthritis,My neck has been very tight and my muscles hav...
...,...,...
235,diabetes,"My emotions fluctuate, and it's hard for me to..."
236,Chicken pox,I have lost my appetite completely and can't s...
237,Dimorphic Hemorrhoids,I've been constipated and it's really hard to ...
238,Bronchial Asthma,must confess that I've been experiencing shor...


## 4. Data Preprocessing - Removing Stopwords

In [12]:
with open("/kaggle/input/medical-federated1/clinical-stopwords.txt", 'r') as f:
    stopwords = set(f.read().splitlines())

In [13]:
def remove_stopwords(text, stopwords):
    return ' '.join([word for word in text.split() if word.lower() not in stopwords])

for df in train_dfs + test_dfs:
    if 'description' in df.columns:
        df['description'] = df['description'].apply(lambda x: remove_stopwords(str(x), stopwords))

## 5. Tokenization with BERT

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_data(examples):
    return tokenizer(examples['description'], padding='max_length', truncation=True, max_length=256)

In [ ]:
def compute_metrics(eval_pred):
    """Returns accuracy + weighted/macro F1 for Trainer evaluation."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    report = classification_report(
        labels, preds,
        target_names=global_label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    return {
        'accuracy':     acc,
        'weighted_f1':  report['weighted avg']['f1-score'],
        'macro_f1':     report['macro avg']['f1-score'],
    }

## 6. Encoding Labels for Classification

In [15]:
all_labels = set()
for df in train_dfs:
    all_labels.update(df['disease'].unique())

global_label_encoder = LabelEncoder()
global_label_encoder.fit(list(all_labels))

for df in train_dfs + test_dfs:
    df['label'] = global_label_encoder.transform(df['disease'])

In [16]:
all_labels

{'Acne',
 'Allergy',
 'Arthritis',
 'Bronchial Asthma',
 'Cervical Spondylosis',
 'Cervical spondylosis',
 'Chicken Pox',
 'Chicken pox',
 'Common Cold',
 'Dengue',
 'Diabetes',
 'Dimorphic Hemorrhoids',
 'Drug Reaction',
 'Fungal Infection',
 'Fungal infection',
 'Gastroesophageal Reflux Disease',
 'Hypertension',
 'Impetigo',
 'Jaundice',
 'Malaria',
 'Migraine',
 'Peptic Ulcer Disease',
 'Pneumonia',
 'Psoriasis',
 'Typhoid',
 'Urinary Tract Infection',
 'Varicose Veins',
 'acne',
 'allergy',
 'back pain',
 'blurry vision',
 'body feels weak',
 'cough',
 'diabetes',
 'drug reaction',
 'ear ache',
 'emotional pain',
 'feeling cold',
 'feeling dizzy',
 'foot ache',
 'gastroesophageal reflux disease',
 'hair falling out',
 'hard to breath',
 'head ache',
 'heart hurts',
 'infected wound',
 'injury from sports',
 'internal pain',
 'joint pain',
 'knee pain',
 'muscle pain',
 'neck pain',
 'open wound',
 'peptic ulcer disease',
 'shoulder pain',
 'skin issue',
 'stomach ache',
 'urinary 

## 7. Handling Imbalanced Data using SMOTE

In [ ]:
USE_DIRICHLET    = True
DIRICHLET_ALPHA  = 0.5   # lower → more skewed non-IID


def dirichlet_partition(dfs, n_clients, alpha, seed=42):
    """
    Combine all client dataframes and re-partition by Dirichlet(alpha).
    Each class's rows are split across clients proportionally to a
    Dirichlet draw, producing realistic label skew.
    Returns a list of n_clients dataframes.
    """
    rng = np.random.default_rng(seed)
    combined = (
        pd.concat(dfs, ignore_index=True)
          .sample(frac=1, random_state=seed)
          .reset_index(drop=True)
    )
    client_indices = [[] for _ in range(n_clients)]

    for cls in combined['label'].unique():
        cls_idx      = combined.index[combined['label'] == cls].tolist()
        proportions  = rng.dirichlet([alpha] * n_clients)
        splits       = (np.cumsum(proportions) * len(cls_idx)).astype(int)
        prev = 0
        for cid, split in enumerate(splits):
            client_indices[cid].extend(cls_idx[prev:split])
            prev = split

    return [
        combined.iloc[idx].reset_index(drop=True)
        for idx in client_indices
    ]


if USE_DIRICHLET:
    print(f"Dirichlet partitioning  α={DIRICHLET_ALPHA}")
    partitioned_dfs = dirichlet_partition(train_dfs, n_clients=3, alpha=DIRICHLET_ALPHA)

    for i, df_p in enumerate(partitioned_dfs):
        n_diseases = df_p['disease'].nunique()
        print(f"  Client {i+1}: {len(df_p):5d} samples  |  {n_diseases} unique diseases")

    # Rebuild tokenised HuggingFace datasets from the new splits
    train_datasets = []
    for df_p in partitioned_dfs:
        ds = Dataset.from_pandas(df_p).map(tokenize_data, batched=True)
        ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
        train_datasets.append(ds)

    active_train_dfs = partitioned_dfs   # used for dataset_sizes below
    print("Non-IID datasets ready.")
else:
    active_train_dfs = train_dfs
    print("Using original IID dataset splits.")

## 7a. Non-IID Data Partitioning — Dirichlet (Phase 3)

Real hospitals have heavy **label skew**: a cardiology ward sees very few skin diseases; a dermatology clinic sees almost no cardiac cases.  
`dirichlet_partition(alpha)` re-distributes the combined training data across clients using a Dirichlet(α) draw per class.

| α | Distribution |
|---|---|
| 0.1 | Highly skewed — each client owns mostly 1–2 diseases |
| 0.5 | Moderately skewed (default) |
| 100 | Nearly IID |

`USE_DIRICHLET = False` keeps the original dataset splits.

In [8]:
train_datasets, test_datasets = [], []
for i in range(3):
    smote = SMOTE(random_state=42)
    _, y_resampled = smote.fit_resample(np.zeros((len(train_dfs[i]), 1)), train_dfs[i]['label'])  
    
    resampled_indices = []
    for class_label in np.unique(y_resampled):
        class_indices = np.where(train_dfs[i]['label'] == class_label)[0]
        class_resampled_indices = np.random.choice(class_indices, size=(y_resampled == class_label).sum(), replace=True)
        resampled_indices.extend(class_resampled_indices)

    train_dataset = Dataset.from_pandas(train_dfs[i]).map(tokenize_data, batched=True)
    test_dataset = Dataset.from_pandas(test_dfs[i]).map(tokenize_data, batched=True)

    train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

    train_datasets.append(train_dataset)
    test_datasets.append(test_dataset)

Map:   0%|          | 0/853 [00:00<?, ? examples/s]

Map:   0%|          | 0/212 [00:00<?, ? examples/s]

Map:   0%|          | 0/5328 [00:00<?, ? examples/s]

Map:   0%|          | 0/667 [00:00<?, ? examples/s]

Map:   0%|          | 0/960 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

In [ ]:
USE_FEDPROX  = True
FEDPROX_MU   = 0.01   # proximal coefficient — 0 = plain FedAvg


class FedProxTrainer(Trainer):
    """Trainer subclass that adds the FedProx proximal term to every batch loss."""

    def __init__(self, *args, global_model=None, mu=0.01, **kwargs):
        super().__init__(*args, **kwargs)
        self.mu = mu
        # Detach and freeze a snapshot of the global parameters
        self._global_params = (
            [p.detach().clone() for p in global_model.parameters()]
            if global_model is not None else None
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        loss    = outputs.loss

        if self._global_params is not None:
            prox = sum(
                (p - p_g).norm(2) ** 2
                for p, p_g in zip(model.parameters(), self._global_params)
            )
            loss = loss + (self.mu / 2) * prox

        return (loss, outputs) if return_outputs else loss


if USE_FEDPROX:
    # Initialise fresh client models to anchor against
    fedprox_base_models = [
        DistilBertForSequenceClassification.from_pretrained(
            'distilbert-base-uncased', num_labels=num_labels
        )
        for _ in range(3)
    ]

    # Use a shared global model as the proximal anchor (random init for round 0)
    anchor_global = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=num_labels
    )

    fedprox_trainers = [
        FedProxTrainer(
            model=fedprox_base_models[i],
            args=training_args,
            train_dataset=train_datasets[i],
            eval_dataset=test_datasets[i],
            processing_class=tokenizer,
            compute_metrics=compute_metrics,
            global_model=anchor_global,
            mu=FEDPROX_MU,
        )
        for i in range(3)
    ]

    print(f"FedProx trainers ready  (μ={FEDPROX_MU})")
    for i, trainer in enumerate(fedprox_trainers):
        print(f"  Training client {i+1} …")
        trainer.train()

    # Downstream aggregation uses these models
    models = [t.model for t in fedprox_trainers]
    print("FedProx training complete.")
else:
    print("USE_FEDPROX=False — using plain / DP-trained models.")

## 9b. FedProx Training (Phase 3)

FedProx adds a **proximal penalty** `(μ/2)‖w − w_global‖²` to each client's local loss, preventing model weights from drifting too far from the global model during local training.  
This is the main fix for non-IID data — client drift is the reason plain FedAvg degrades under heterogeneous label distributions.

`FedProxTrainer` is a minimal `Trainer` subclass that overrides `compute_loss` to inject the proximal term.  
`FEDPROX_MU = 0` degrades exactly to plain FedAvg.

num_labels = len(global_label_encoder.classes_)

# DistilBERT is 40% smaller and 60% faster than BERT with ~97% accuracy retention
models = [
    DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=num_labels)
    for _ in range(3)
]

training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=8,
    num_train_epochs=15,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy='no',
    eval_strategy='epoch',
    logging_strategy='epoch',
    fp16=True,
    load_best_model_at_end=False,
)

torch.cuda.empty_cache()

trainers = [
    Trainer(
        model=models[i],
        args=training_args,
        train_dataset=train_datasets[i],
        eval_dataset=test_datasets[i],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )
    for i in range(3)
]

In [9]:
num_labels = len(global_label_encoder.classes_)
models = [BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels) for _ in range(3)]

training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=8,  
    num_train_epochs=15,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy='no',
    logging_strategy='no',
    fp16=True  
)

torch.cuda.empty_cache()  

trainers = [
    Trainer(
        model=models[i],
        args=training_args,
        train_dataset=train_datasets[i],
        eval_dataset=test_datasets[i],
        tokenizer=tokenizer
    )
    for i in range(3)
]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to cont

TOP_K_SPARSITY = 0.99   # keep top 1% of update values; 0 = no compression


def top_k_sparsify(tensor, sparsity):
    """Zero out all but the top-(1-sparsity) fraction of values by magnitude."""
    if sparsity <= 0 or tensor.numel() == 0:
        return tensor
    k         = max(1, int((1 - sparsity) * tensor.numel()))
    threshold = tensor.abs().flatten().topk(k).values[-1]
    return tensor * (tensor.abs() >= threshold)


def federated_averaging(local_models, dataset_sizes,
                        ref_global=None, sparsity=TOP_K_SPARSITY):
    """
    Weighted FedAvg with optional Top-k gradient compression.

    If ref_global is provided:
        delta_i  = local_params_i − global_params
        compressed_delta_i = top_k(delta_i, sparsity)
        new_global = global + Σ weight_i · compressed_delta_i
    Otherwise:
        plain weighted average of raw weights (original behaviour).
    """
    total_data = sum(dataset_sizes)

    if ref_global is not None:
        g_params   = {n: p.data.clone() for n, p in ref_global.named_parameters()}
        avg_params = {n: p.data.clone() for n, p in ref_global.named_parameters()}

        n_params_sent = 0
        n_params_total = sum(p.numel() for p in ref_global.parameters())

        for model, data_size in zip(local_models, dataset_sizes):
            weight = data_size / total_data
            for name, param in model.named_parameters():
                delta      = param.data - g_params[name]
                compressed = top_k_sparsify(delta, sparsity)
                avg_params[name] += weight * compressed
                n_params_sent += int((compressed != 0).sum().item())

        ratio = n_params_sent / (n_params_total * len(local_models)) * 100
        print(f"  Compression: transmitted {ratio:.1f}% of parameters per client "
              f"(sparsity={sparsity})")
    else:
        avg_params = {n: torch.zeros_like(p) for n, p in local_models[0].named_parameters()}
        for model, data_size in zip(local_models, dataset_sizes):
            weight = data_size / total_data
            for name, param in model.named_parameters():
                avg_params[name] += weight * param.data

    new_global = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=num_labels
    )
    state = new_global.state_dict()
    state.update(avg_params)
    new_global.load_state_dict(state)
    return new_global


dataset_sizes = [len(df) for df in active_train_dfs]
global_model  = federated_averaging(
    models, dataset_sizes,
    ref_global=anchor_global if USE_FEDPROX else None,
    sparsity=TOP_K_SPARSITY,
)

In [ ]:
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from torch.utils.data import DataLoader
from torch.optim import AdamW

USE_DP         = True    # set False to keep using plain Trainer models above
TARGET_EPSILON = 8.0     # ε budget per client per round
DELTA          = 1e-5
MAX_GRAD_NORM  = 1.0     # L2 clipping bound
DP_EPOCHS      = 15
DP_BATCH_SIZE  = 8


def _collate(batch):
    return {
        'input_ids':      torch.stack([b['input_ids']      for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'labels':         torch.stack([b['label']          for b in batch]),
    }


def train_client_dp(client_idx, train_dataset, target_epsilon, delta,
                    max_grad_norm, epochs, batch_size):
    """Train one federated client with Opacus DP-SGD."""
    base = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=num_labels
    )
    # Replace multi-head attention with DP-safe implementation
    model = ModuleValidator.fix(base)
    model.train()

    loader    = DataLoader(train_dataset, batch_size=batch_size,
                           shuffle=True, collate_fn=_collate)
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

    privacy_engine = PrivacyEngine()
    model, optimizer, loader = privacy_engine.make_private_with_epsilon(
        module=model,
        optimizer=optimizer,
        data_loader=loader,
        target_epsilon=target_epsilon,
        target_delta=delta,
        max_grad_norm=max_grad_norm,
        epochs=epochs,
    )

    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(dev)

    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for batch in loader:
            optimizer.zero_grad()
            out  = model(
                input_ids=batch['input_ids'].to(dev),
                attention_mask=batch['attention_mask'].to(dev),
                labels=batch['labels'].to(dev),
            )
            out.loss.backward()
            optimizer.step()
            epoch_loss += out.loss.item()

        if epoch % 5 == 0 or epoch == epochs:
            eps = privacy_engine.get_epsilon(delta)
            print(f"  Client {client_idx} | epoch {epoch:2d}/{epochs} "
                  f"| loss {epoch_loss/len(loader):.4f} | ε={eps:.2f}")

    final_eps = privacy_engine.get_epsilon(delta)
    print(f"→ Client {client_idx} done  ε={final_eps:.3f} (budget={target_epsilon})")
    return model, final_eps


if USE_DP:
    print("Training all clients with Opacus DP-SGD …\n")
    dp_models, dp_epsilons = [], []
    for i in range(3):
        print(f"[Client {i+1}]")
        m, eps = train_client_dp(
            i + 1, train_datasets[i],
            target_epsilon=TARGET_EPSILON, delta=DELTA,
            max_grad_norm=MAX_GRAD_NORM,
            epochs=DP_EPOCHS, batch_size=DP_BATCH_SIZE,
        )
        dp_models.append(m)
        dp_epsilons.append(eps)

    # Use DP models in all downstream cells
    models = dp_models
    print(f"\nAll clients trained.  ε budgets: {[round(e, 2) for e in dp_epsilons]}")
else:
    print("USE_DP=False — using plain Trainer models from Section 9.")

## 9a. DP-SGD Training with Opacus (Phase 2)

Replaces the plain HuggingFace Trainer above with a proper **DP-SGD training loop** using [Opacus](https://opacus.ai/).

Key changes vs plain training:
- `ModuleValidator.fix(model)` swaps DistilBERT's multi-head attention for a DP-compatible version
- `PrivacyEngine.make_private_with_epsilon()` wraps the optimizer and DataLoader to enforce per-sample gradient clipping and Gaussian noise addition
- `privacy_engine.get_epsilon(delta)` tracks cumulative ε after every epoch

Set `USE_DP = True` to use these models in all subsequent cells instead of the plain Trainer models.

In [10]:
for trainer in trainers:
    trainer.train()

Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.


Step,Training Loss


Step,Training Loss


Step,Training Loss


print("=" * 65)
print("Per-Client Evaluation (DistilBERT)")
print("=" * 65)

for i, trainer in enumerate(trainers):
    predictions = trainer.predict(test_datasets[i])
    preds = np.argmax(predictions.predictions, axis=1)
    true  = test_dfs[i]['label'].values

    acc = accuracy_score(true, preds)
    report = classification_report(
        true, preds,
        target_names=global_label_encoder.classes_,
        zero_division=0,
    )
    print(f"\nClient {i+1}  Accuracy: {acc:.4f}")
    print(report)

In [11]:
for i, trainer in enumerate(trainers):
    predictions = trainer.predict(test_datasets[i])
    preds = np.argmax(predictions.predictions, axis=1)
    acc = accuracy_score(test_dfs[i]['label'], preds)
    print(f'Model {i+1} Test Accuracy: {acc:.4f}')

Model 1 Test Accuracy: 0.9387


Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.


Model 2 Test Accuracy: 1.0000


Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.


Model 3 Test Accuracy: 0.9875


def federated_averaging(models, dataset_sizes):
    total_data = sum(dataset_sizes)
    avg_params = {name: torch.zeros_like(param) for name, param in models[0].named_parameters()}

    for model, data_size in zip(models, dataset_sizes):
        weight = data_size / total_data
        for name, param in model.named_parameters():
            avg_params[name] += weight * param.data

    global_model = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=num_labels
    )
    state = global_model.state_dict()
    state.update(avg_params)
    global_model.load_state_dict(state)
    return global_model

In [12]:
def federated_averaging(models, dataset_sizes):
    
    num_models = len(models)
    total_data = sum(dataset_sizes)
    avg_params = {name: torch.zeros_like(param) for name, param in models[0].named_parameters()}

    for model, data_size in zip(models, dataset_sizes):
        weight = data_size / total_data  
        for name, param in model.named_parameters():
            avg_params[name] += weight * param.data  

    global_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)
    global_state_dict = global_model.state_dict()
    
    for name in avg_params:
        global_state_dict[name] = avg_params[name]
    
    global_model.load_state_dict(global_state_dict)
    
    return global_model

In [13]:
dataset_sizes = [len(train_dfs[0]), len(train_dfs[1]), len(train_dfs[2])]
global_model = federated_averaging(models, dataset_sizes)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
global_model.to(device)
global_model.eval()

torch.cuda.empty_cache()  

all_preds, all_labels = [], []
batch_size = 4  

In [15]:
with torch.no_grad():
    for test_dataset, test_df in zip(test_datasets, test_dfs):
        for i in range(0, len(test_dataset), batch_size):
            batch = {key: test_dataset[key][i:i+batch_size].to(device) for key in ['input_ids', 'attention_mask']}
            labels_batch = test_dataset['label'][i:i+batch_size].to(device)
            
            outputs = global_model(**batch)
            preds = np.argmax(outputs.logits.cpu().numpy(), axis=1)
            
            all_preds.extend(preds)
            all_labels.extend(labels_batch.cpu().numpy())

## 12. Evaluating Global Model

In [ ]:
print("=" * 65)
print("Global Federated Model Evaluation")
print("=" * 65)
print(f"Accuracy: {global_accuracy:.4f}")

global_report = classification_report(
    all_labels, all_preds,
    target_names=global_label_encoder.classes_,
    zero_division=0,
)
print("\nDetailed Classification Report:")
print(global_report)

In [ ]:
# Theoretical ε/δ privacy budget analysis using Opacus RDP accountant.
# This shows what ε would be if DP-SGD were applied with these hyperparameters.
# Phase 2 wires in the actual DP-SGD training loop to make these guarantees real.
try:
    from opacus.accountants import RDPAccountant

    NOISE_MULTIPLIER = 1.1   # σ  (Gaussian noise scale)
    DELTA            = 1e-5
    BATCH_SIZE       = 8
    EPOCHS           = 15

    print(f"Theoretical Privacy Budget  (σ={NOISE_MULTIPLIER}, δ={DELTA})")
    print("=" * 55)
    for i, train_df in enumerate(train_dfs):
        n           = len(train_df)
        sample_rate = BATCH_SIZE / n
        steps       = int(EPOCHS * (n / BATCH_SIZE))

        accountant = RDPAccountant()
        for _ in range(steps):
            accountant.step(noise_multiplier=NOISE_MULTIPLIER, sample_rate=sample_rate)
        eps = accountant.get_epsilon(delta=DELTA)

        print(f"Client {i+1}: n={n:5d}  steps={steps:5d}  ε = {eps:6.2f}")

    print("\nLower ε → stronger privacy guarantee.")
    print("ε < 10 is generally considered acceptable for medical data.")
    print("Enable full DP-SGD in Phase 2 to enforce these bounds during training.")

except ImportError:
    print("Install opacus to compute privacy budget:  pip install opacus>=1.4.0")

## 13a. Theoretical Privacy Budget Estimate

In [19]:
print(f'Global Model Test Accuracy: {global_accuracy:.4f}')

Global Model Test Accuracy: 0.9301


## 13. Explainable AI - SHAP Analysis

In [20]:
def explain_prediction(model, tokenizer, text, label_encoder):
    if isinstance(text, pd.Series):
        text = text.astype(str).tolist()
    elif isinstance(text, str):
        text = [text]

    id2label = {i: label for i, label in enumerate(label_encoder.classes_)}

    def model_wrapper(input_texts):
        if isinstance(input_texts, np.ndarray):
            input_texts = input_texts.tolist()
        elif isinstance(input_texts, str):
            input_texts = [input_texts]
        elif not isinstance(input_texts, list) or not all(isinstance(t, str) for t in input_texts):
            raise ValueError("Input must be a string or a list of strings.")

        tokenized = tokenizer(input_texts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            logits = model(**tokenized).logits
        return logits.cpu().numpy()

    masker = shap.maskers.Text(tokenizer)
    explainer = shap.Explainer(model_wrapper, masker)
    shap_values = explainer(text)

    shap_values.output_names = [id2label[i] for i in range(len(id2label))]

    shap.text_plot(shap_values)

## 14. Model Prediction on Sample Data

In [21]:
test_text = str(test_dfs[0]['description'].iloc[0])
prediction = global_model(**tokenizer(test_text, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device))
pred_label = torch.argmax(prediction.logits, dim=1).cpu().numpy()[0]
pred_disease = global_label_encoder.inverse_transform([pred_label])[0]
print(test_text)
print(f'Predicted Disease: {pred_disease}')

burning sensation stomach comes goes. It's worse eat lie down. heartburn indigestion.
Predicted Disease: stomach ache


In [ ]:
torch.save(global_model.state_dict(), "global_model1.pt")
tokenizer.save_pretrained("tokenizer")
joblib.dump(global_label_encoder, "label_encoder.pkl")
print("Saved: global_model1.pt | tokenizer/ | label_encoder.pkl")

## 15. Saving the Model and Artifacts for Flask

In [23]:
import joblib

In [24]:
torch.save(global_model.state_dict(), "global_model.pt")

tokenizer.save_pretrained("tokenizer")

joblib.dump(global_label_encoder, "label_encoder.pkl")

['label_encoder.pkl']

# References

https://www.kaggle.com/datasets/dpm3333/patient-symptoms-report-image-and-disease-dataset/
https://github.com/kavgan/clinical-concepts